# Data Preprocessing and Modeling Pipeline 


This notebook covers the full workflow for training CNN models from scratch to classify paddy varieties. 
The steps include:
- Data cleaning and preprocessing
- Handling class imbalance with oversampling and class weights
- Data augmentation and generator setup
- Defining CNN architectures (MobileNetV3-Small, EfficientNetB0, ResNet18, MobileNetV2-Lite)
- Hyperparameter tuning with Optuna
- Retraining the best models
- Evaluation and metric visualization
- Generating predictions for test data

In [ ]:
# Install necessary packages for the notebook environment (run once)
# %pip install numpy pandas matplotlib seaborn scikit-learn Pillow tensorflow optuna optuna-integration opencv-python

## 1. Importing necessary libraries and modules

In [ ]:
# Import all necessary libraries and modules for data handling, visualization, modeling, and evaluation
import os
import random
import json
import shutil
import cv2
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from math import ceil
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, Model, regularizers
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, Conv2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf
import optuna
from optuna.integration import TFKerasPruningCallback
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_slice
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from tensorflow.keras.models import load_model

## 2. Parameters & Random Seed Configuration

In [3]:
# Define configuration parameters for reproducibility, input sizes, batch sizes, and training settings.

CONFIG = {
    'SEED': 42,
    'INPUT_SIZE': (224, 224),  
    'BATCH_SIZE': 32,
    'TEST_SIZE': 0.2,  # Validation split ratio
    'IMAGE_ORIGINAL': 'train_images',  # Original dataset folder
    'IMAGE_DIR_PROCESSED': 'train_images_copy',  # Copy for preprocessing modifications
    'CSV_PATH': 'meta_train.csv',  # Metadata CSV file path
    'MODEL_DIR': 'saved_models/',  # Where to save trained models
    'HISTORY_DIR': 'saved_histories/',
    'EXPERIMENT_DIR': 'experiments/',
    'EPOCHS': 100,  # Retraining epochs
    'EPOCHS_TUNING': 30,  # Tuning epochs for Optuna
    'OPTUNA_TRIALS': 20,
    'LR_MIN': 1e-5,
    'LR_MAX': 1e-2,
    'EARLY_STOP_PATIENCE': 10,
    'REDUCE_LR_PATIENCE': 3,
}

# Set random seeds for reproducibility
random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

## 3.Data Preprocessing

### 3.1. Load & Clean Metadata

In [4]:
# Prepare Processed Image Directory ---
if not os.path.exists(CONFIG['IMAGE_DIR_PROCESSED']):
    shutil.copytree(CONFIG['IMAGE_ORIGINAL'], CONFIG['IMAGE_DIR_PROCESSED'])
    print(f"Copied images from '{CONFIG['IMAGE_ORIGINAL']}' to '{CONFIG['IMAGE_DIR_PROCESSED']}'.")
else:
    print(f"Folder '{CONFIG['IMAGE_DIR_PROCESSED']}' already exists; using it.")

Folder 'train_images_copy' already exists; using it.


### 3.2. Image Orientation Fix

In [5]:
# Fix Image Orientation in Processed Directory ---
def fix_image_orientation(directory, expected_width=480, expected_height=640):
    for subdir, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                path = os.path.join(subdir, file)
                img = cv2.imread(path)
                if img is None:
                    continue
                h, w = img.shape[:2]
                if w == 640 and h == 480:
                    # Rotate 90 degrees clockwise to convert to 480x640.
                    img_rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
                    cv2.imwrite(path, img_rotated)
                    print(f"Rotated {path} to {expected_width}x{expected_height}")

# Run the orientation fix on the processed images directory.
fix_image_orientation(CONFIG['IMAGE_DIR_PROCESSED'], expected_width=480, expected_height=640)

In [6]:
# Load and Clean the Metadata 
df = pd.read_csv(CONFIG['CSV_PATH'])

# Remove any leading/trailing whitespace from key columns.
df['image_id'] = df['image_id'].str.strip()
df['label'] = df['label'].str.strip()
df['variety'] = df['variety'].str.strip()

# create filepath accordingly.
df['filepath'] = df['label'] + "/" + df['image_id']

# Save the cleaned metadata to file for reproducibility.
df.to_csv("meta_train_copy.csv", index=False)
print("Cleaned CSV saved as 'meta_train_copy.csv'")
print(df.head())

Cleaned CSV saved as 'meta_train_copy.csv'
     image_id                  label variety  age  \
0  100330.jpg  bacterial_leaf_blight   ADT45   45   
1  100365.jpg  bacterial_leaf_blight   ADT45   45   
2  100382.jpg  bacterial_leaf_blight   ADT45   45   
3  100632.jpg  bacterial_leaf_blight   ADT45   45   
4  101918.jpg  bacterial_leaf_blight   ADT45   45   

                           filepath  
0  bacterial_leaf_blight/100330.jpg  
1  bacterial_leaf_blight/100365.jpg  
2  bacterial_leaf_blight/100382.jpg  
3  bacterial_leaf_blight/100632.jpg  
4  bacterial_leaf_blight/101918.jpg  


In [7]:
# Explore and Verify the Variety Distribution 
variety_counts = df.groupby("variety")["image_id"].count().reset_index()
variety_counts.columns = ["Variety", "Number of Images"]
print("\nVariety Distribution:")
print(variety_counts)


Variety Distribution:
          Variety  Number of Images
0           ADT45              6992
1      AndraPonni               377
2    AtchayaPonni               461
3            IR20               114
4  KarnatakaPonni               988
5        Onthanel               351
6           Ponni               657
7              RR                36
8           Surya                32
9           Zonal               399


### 3.3. Train/Validation Split and Oversampling

In [8]:
# Train/Validation Split (Stratify on 'variety') 
train_df, val_df = train_test_split(
    df,
    test_size=CONFIG['TEST_SIZE'],
    stratify=df['variety'],
    random_state=CONFIG['SEED']
)
print("\n[INFO] Train/Val split done.")
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")


[INFO] Train/Val split done.
Train size: 8325
Validation size: 2082


In [9]:
# Define oversampling criteria: For classes with fewer than 500 images.
oversample_threshold = 500
max_oversample_factor = 10  

oversampled_train_df_list = []
for cls in train_df['variety'].unique():
    cls_df = train_df[train_df['variety'] == cls]
    count = len(cls_df)
    if count < oversample_threshold:
        factor = min(int(ceil(oversample_threshold / count)), max_oversample_factor)
        n_samples = count * factor
        print(f"Oversampling class '{cls}': original count = {count}; replicating {factor}× to reach ~{n_samples} samples.")
        oversampled_cls_df = cls_df.sample(n=n_samples, replace=True, random_state=CONFIG['SEED'])
    else:
        oversampled_cls_df = cls_df
    oversampled_train_df_list.append(oversampled_cls_df)

oversampled_train_df = pd.concat(oversampled_train_df_list).reset_index(drop=True)
print("\nOversampled Train Distribution:")
print(oversampled_train_df.groupby('variety')['image_id'].count())

Oversampling class 'AndraPonni': original count = 302; replicating 2× to reach ~604 samples.
Oversampling class 'AtchayaPonni': original count = 369; replicating 2× to reach ~738 samples.
Oversampling class 'Zonal': original count = 319; replicating 2× to reach ~638 samples.
Oversampling class 'IR20': original count = 91; replicating 6× to reach ~546 samples.
Oversampling class 'Onthanel': original count = 281; replicating 2× to reach ~562 samples.
Oversampling class 'RR': original count = 29; replicating 10× to reach ~290 samples.
Oversampling class 'Surya': original count = 26; replicating 10× to reach ~260 samples.

Oversampled Train Distribution:
variety
ADT45             5593
AndraPonni         604
AtchayaPonni       738
IR20               546
KarnatakaPonni     790
Onthanel           562
Ponni              525
RR                 290
Surya              260
Zonal              638
Name: image_id, dtype: int64


In [10]:
# Extract the target labels from the oversampled training data.
y_over = oversampled_train_df['variety']

# Get the unique classes (varieties) present after oversampling.
classes_over = np.unique(y_over)

# Compute class weights using a 'balanced' strategy 
raw_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=classes_over,
    y=y_over
)

# Create a dictionary mapping each class label to its computed weight.
class_weights_oversampled = dict(zip(classes_over, raw_weights))
print("Raw Class Weights After Oversampling:")
print(class_weights_oversampled)

Raw Class Weights After Oversampling:
{'ADT45': 0.18855712497765065, 'AndraPonni': 1.7460264900662252, 'AtchayaPonni': 1.4289972899728998, 'IR20': 1.9315018315018315, 'KarnatakaPonni': 1.3349367088607595, 'Onthanel': 1.8765124555160142, 'Ponni': 2.0087619047619047, 'RR': 3.636551724137931, 'Surya': 4.056153846153846, 'Zonal': 1.6529780564263323}


### 3.4. Data Augmentation & Generators

In [11]:
# Data Augmentation & Generators 
# Define augmentation for training; validation set will only be rescaled.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    # validation_split=CONFIG['TEST_SIZE']  
)
val_datagen = ImageDataGenerator(rescale=1./255)

# Create training generator using the oversampled training DataFrame.
train_generator = train_datagen.flow_from_dataframe(
    dataframe=oversampled_train_df,
    directory=CONFIG['IMAGE_DIR_PROCESSED'],
    x_col="filepath",
    y_col="variety",
    target_size=(CONFIG['INPUT_SIZE'][0], CONFIG['INPUT_SIZE'][1]),
    batch_size=CONFIG['BATCH_SIZE'],
    class_mode='categorical',  # For multi-class classification.
    shuffle=True,
    seed=CONFIG['SEED']
)

# Create validation generator using the original validation DataFrame.
valid_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=CONFIG['IMAGE_DIR_PROCESSED'],
    x_col="filepath",
    y_col="variety",target_size=(CONFIG['INPUT_SIZE'][0], CONFIG['INPUT_SIZE'][1]),
    batch_size=CONFIG['BATCH_SIZE'],
    class_mode='categorical',
    shuffle=False,
    seed=CONFIG['SEED']
)

print(f"\n[INFO] Training samples: {train_generator.samples}")
print(f"[INFO] Validation samples: {valid_generator.samples}")
print(f"[INFO] Class mapping: {train_generator.class_indices}")

Found 10546 validated image filenames belonging to 10 classes.
Found 2082 validated image filenames belonging to 10 classes.

[INFO] Training samples: 10546
[INFO] Validation samples: 2082
[INFO] Class mapping: {'ADT45': 0, 'AndraPonni': 1, 'AtchayaPonni': 2, 'IR20': 3, 'KarnatakaPonni': 4, 'Onthanel': 5, 'Ponni': 6, 'RR': 7, 'Surya': 8, 'Zonal': 9}


In [12]:
# Assuming 'mapping' is obtained from your training generator:
mapping = train_generator.class_indices  # e.g., {'ADT45': 0, 'AndraPonni': 1, ...}

# Map the scaled weights to integer keys.
# Option 1: When not use Step 2: Scale Down the Extreme Weights (Optional Smoothing)
new_class_weights = {mapping[k]: v for k, v in class_weights_oversampled.items()}
print("\nFinal Class Weights (integer keys):")
print(new_class_weights)


Final Class Weights (integer keys):
{0: 0.18855712497765065, 1: 1.7460264900662252, 2: 1.4289972899728998, 3: 1.9315018315018315, 4: 1.3349367088607595, 5: 1.8765124555160142, 6: 2.0087619047619047, 7: 3.636551724137931, 8: 4.056153846153846, 9: 1.6529780564263323}


## 4. Modeling

### 4.1. MobileNetV3-Small

##### 4.1.1 MobileNetV3-Small Definition

In [ ]:
# Define the MobileNetV3 Small architecture
def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

def se_block(inputs, se_ratio=0.25):
    filters = inputs.shape[-1]
    reduced_filters = max(1, int(filters * se_ratio))
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Reshape((1, 1, filters))(x)
    x = layers.Conv2D(reduced_filters, 1, activation='relu')(x)
    x = layers.Conv2D(filters, 1, activation='hard_sigmoid')(x)
    return layers.Multiply()([inputs, x])

def mbv3_block(inputs, out_channels, expansion, stride, se, activation, block_id):
    in_channels = inputs.shape[-1]
    x = inputs

    if expansion != 1:
        x = layers.Conv2D(in_channels * expansion, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    if se:
        x = se_block(x)

    x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and in_channels == out_channels:
        x = layers.Add()([inputs, x])

    return x

def build_mobilenetv3_small_from_trial(trial, input_shape, num_classes):
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units = trial.suggest_int("dense_units", 128, 512)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)

    activation = hard_swish
    inputs = Input(shape=input_shape, name="input")

    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = mbv3_block(x, 16, 1, 2, True, 'relu', 0)
    x = mbv3_block(x, 24, 4, 2, False, 'relu', 1)
    x = mbv3_block(x, 24, 3, 1, False, 'relu', 2)
    x = mbv3_block(x, 40, 4, 2, True, activation, 3)
    x = mbv3_block(x, 40, 6, 1, True, activation, 4)
    x = mbv3_block(x, 48, 6, 1, True, activation, 5)
    x = mbv3_block(x, 96, 6, 2, True, activation, 6)
    x = mbv3_block(x, 96, 6, 1, True, activation, 7)

    x = layers.Conv2D(576, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(dense_units, activation=activation)(x)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs=inputs, outputs=outputs, name="MobileNetV3_Small_Scratch")
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


##### 4.1.2. MobileNetV3-Small Optuna Tuning

In [ ]:
# Objective function for Optuna
def objective_mbv3(trial):
    input_shape = CONFIG['INPUT_SIZE'] + (3,)
    num_classes = len(train_generator.class_indices) 
    model = build_mobilenetv3_small_from_trial(trial, input_shape, num_classes)

    study_dir = "experiments/MobileNetV3-Small/tuning"
    checkpoint_dir = os.path.join(study_dir, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_path = os.path.join(checkpoint_dir, f"trial_{trial.number}.h5")
    checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=0
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True),
        TFKerasPruningCallback(trial, monitor="val_accuracy"),
        checkpoint_callback
    ]

    history = model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=CONFIG['EPOCHS_TUNING'],
        callbacks=callbacks,
        verbose=1,
        class_weight=new_class_weights,
        workers=8
    )

    val_acc = max(history.history['val_accuracy'])

    tf.keras.backend.clear_session()
    gc.collect()

    return val_acc

In [ ]:
# Study and Optimization
mbv3_result_dir = os.path.join(CONFIG['MODEL_DIR'], "MobileNetV3-Small_Tuned")
os.makedirs(mbv3_result_dir, exist_ok=True)

study_mbv3 = optuna.create_study(
    study_name="MobileNetV3-Small_Tuning",
    direction="maximize",
    sampler=TPESampler(seed=CONFIG['SEED']),
    pruner=MedianPruner(n_warmup_steps=5)
)

study_mbv3.optimize(objective_mbv3, n_trials=CONFIG['OPTUNA_TRIALS'])

[I 2025-05-15 00:09:58,913] A new study created in memory with name: MobileNetV3-Small_Tuning
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\2755267092.py:40: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Epoch 1/30
330/330 [==============================] - 16s 37ms/step - loss: 1.7240 - accuracy: 0.2310 - val_loss: 2.8480 - val_accuracy: 0.0360
Epoch 2/30
330/330 [==============================] - 12s 36ms/step - loss: 1.2327 - accuracy: 0.3449 - val_loss: 3.3041 - val_accuracy: 0.1734
Epoch 3/30
330/330 [==============================] - 12s 36ms/step - loss: 1.0328 - accuracy: 0.4037 - val_loss: 4.9142 - val_accuracy: 0.1595
Epoch 4/30
330/330 [==============================] - 12s 36ms/step - loss: 0.8772 - accuracy: 0.4902 - val_loss: 3.9466 - val_accuracy: 0.2003
Epoch 5/30
330/330 [==============================] - 13s 38ms/step - loss: 0.6961 - accuracy: 0.5768 - val_loss: 2.4098 - val_accuracy: 0.3237
Epoch 6/30
330/330 [==============================] - 13s 40ms/step - loss: 0.6335 - accuracy: 0.6076 - val_loss: 4.9243 - val_accuracy: 0.3410
Epoch 7/30
330/330 [==============================] - 13s 40ms/step - loss: 0.5354 - accuracy: 0.6510 - val_loss: 1.2152 - val_accuracy:

[I 2025-05-15 00:14:00,713] Trial 0 finished with value: 0.7939481139183044 and parameters: {'dropout_rate': 0.3123620356542087, 'dense_units': 494, 'lr': 0.0029106359131330704}. Best is trial 0 with value: 0.7939481139183044.


Epoch 1/30
330/330 [==============================] - 15s 40ms/step - loss: 1.6515 - accuracy: 0.2592 - val_loss: 2.1591 - val_accuracy: 0.0442
Epoch 2/30
330/330 [==============================] - 13s 38ms/step - loss: 1.1079 - accuracy: 0.4075 - val_loss: 2.0333 - val_accuracy: 0.1446
Epoch 3/30
330/330 [==============================] - 13s 38ms/step - loss: 0.8521 - accuracy: 0.5216 - val_loss: 1.8537 - val_accuracy: 0.3790
Epoch 4/30
330/330 [==============================] - 13s 38ms/step - loss: 0.6786 - accuracy: 0.5941 - val_loss: 1.5316 - val_accuracy: 0.4640
Epoch 5/30
330/330 [==============================] - 13s 39ms/step - loss: 0.5588 - accuracy: 0.6444 - val_loss: 1.1331 - val_accuracy: 0.5903
Epoch 6/30
330/330 [==============================] - 13s 39ms/step - loss: 0.4855 - accuracy: 0.6829 - val_loss: 1.5504 - val_accuracy: 0.4861
Epoch 7/30
330/330 [==============================] - 13s 38ms/step - loss: 0.4069 - accuracy: 0.7141 - val_loss: 1.4858 - val_accuracy:

[I 2025-05-15 00:16:13,129] Trial 1 finished with value: 0.6047070026397705 and parameters: {'dropout_rate': 0.379597545259111, 'dense_units': 188, 'lr': 0.00020511104188433984}. Best is trial 0 with value: 0.7939481139183044.


...

[I 2025-05-15 01:05:59,294] Trial 17 pruned. Trial was pruned at epoch 10.


Epoch 1/30
330/330 [==============================] - 16s 42ms/step - loss: 1.5019 - accuracy: 0.3066 - val_loss: 2.0544 - val_accuracy: 0.0634
Epoch 2/30
330/330 [==============================] - 14s 40ms/step - loss: 0.9782 - accuracy: 0.4525 - val_loss: 1.6447 - val_accuracy: 0.3679
Epoch 3/30
330/330 [==============================] - 13s 39ms/step - loss: 0.7655 - accuracy: 0.5476 - val_loss: 3.8982 - val_accuracy: 0.1734
Epoch 4/30
330/330 [==============================] - 13s 40ms/step - loss: 0.6185 - accuracy: 0.6016 - val_loss: 1.7258 - val_accuracy: 0.4529
Epoch 5/30
330/330 [==============================] - 13s 39ms/step - loss: 0.5159 - accuracy: 0.6499 - val_loss: 2.3410 - val_accuracy: 0.3746
Epoch 6/30
329/330 [============================>.] - ETA: 0s - loss: 0.4474 - accuracy: 0.6820

[I 2025-05-15 01:07:22,589] Trial 18 pruned. Trial was pruned at epoch 5.


Epoch 1/30
330/330 [==============================] - 16s 41ms/step - loss: 1.5757 - accuracy: 0.2744 - val_loss: 2.1719 - val_accuracy: 0.0360
Epoch 2/30
330/330 [==============================] - 13s 39ms/step - loss: 1.0999 - accuracy: 0.4105 - val_loss: 2.2439 - val_accuracy: 0.3079
Epoch 3/30
330/330 [==============================] - 13s 39ms/step - loss: 0.8635 - accuracy: 0.4951 - val_loss: 2.3404 - val_accuracy: 0.2714
Epoch 4/30
330/330 [==============================] - 13s 39ms/step - loss: 0.7100 - accuracy: 0.5705 - val_loss: 2.2327 - val_accuracy: 0.3103
Epoch 5/30
330/330 [==============================] - 13s 39ms/step - loss: 0.5736 - accuracy: 0.6246 - val_loss: 2.5134 - val_accuracy: 0.3213
Epoch 6/30
329/330 [============================>.] - ETA: 0s - loss: 0.5020 - accuracy: 0.6656

[I 2025-05-15 01:08:44,280] Trial 19 pruned. Trial was pruned at epoch 5.


In [ ]:
# Save study results
# Save trials
trials_df = study_mbv3.trials_dataframe()
trials_df.to_csv(os.path.join(mbv3_result_dir, "optuna_trials_mbv3.csv"), index=False)

# Save best trial info
best_trial = study_mbv3.best_trial
best_result = {
    **best_trial.params,
    "best_value": best_trial.value,
    "trial_number": best_trial.number
}
pd.DataFrame([best_result]).to_csv(os.path.join(mbv3_result_dir, "best_params_mbv3.csv"), index=False)

# Save best model weights
best_dir = "experiments/MobileNetV3-Small/tuning"
os.makedirs(best_dir, exist_ok=True)

with open(os.path.join(best_dir, "best_params.json"), "w") as f:
    json.dump(study_mbv3.best_params, f, indent=2)

# Copy best checkpoint
checkpoint_path = os.path.join(best_dir, "checkpoints", f"trial_{best_result['trial_number']}.h5")
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, os.path.join(best_dir, "best_model_mbv3.h5"))
else:
    print(f"Checkpoint not found: {checkpoint_path}")

print(f"\n✅ Tuning complete. Best val_accuracy: {best_result['best_value']:.4f}")
print(f"📄 Saved to: {mbv3_result_dir}")

### 4.2. EfficientNetB0

##### 4.2.1. EfficientNetB0 Definition

In [ ]:
# Define the Efficient Net architecture
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras import Input, Model
from tensorflow.keras.optimizers import Adam

def build_efficientnet_b0_from_trial(trial, input_shape, num_classes):
    # Hyperparameters from Optuna
    dropout_rate = trial.suggest_float("dropout_rate", 0.3, 0.6)
    dense_units = trial.suggest_int("dense_units", 128, 512, step=128)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    
    # EfficientNetB0 Base Model (without pre-trained weights)
    inputs = layers.Input(shape=input_shape, name="input_layer")
    
    # Initial Conv Block (stem)
    x = layers.Conv2D(32, 3, strides=2, padding='same', activation='swish', name="conv_block")(inputs)
    
    # Depthwise separable convolution blocks (EfficientNet architecture)
    for i in range(3):
        x_res = x
        x = layers.DepthwiseConv2D(3, padding='same', activation='swish', name=f'depthwise_conv2d_{i}')(x)
        x = layers.Conv2D(32, 1, activation='swish', name=f'conv2d_{i}')(x)
        if x.shape == x_res.shape:
            x = layers.Add(name=f'add_{i}')([x, x_res])
    
    # Global Average Pooling
    x = layers.GlobalAveragePooling2D(name='global_avg_pooling')(x)
    
    # Dense layer
    x = layers.Dense(dense_units, activation='swish', name='dense_1')(x)
    x = layers.Dropout(dropout_rate, name='dropout_1')(x)
    
    # Output layer (softmax for multi-class classification)
    outputs = layers.Dense(num_classes, activation='softmax', name='output_layer')(x)
    
    model = models.Model(inputs, outputs)

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


##### 4.2.2. EfficientNetB0 Optuna Tuning

In [ ]:
# Optuna Objective Function
import optuna
from tensorflow.keras.callbacks import EarlyStopping
from optuna.integration import TFKerasPruningCallback
from tensorflow.keras.callbacks import ModelCheckpoint
import os

def objective_efficientnet_b0(trial):
    # Unpack input size correctly
    input_height, input_width = CONFIG['INPUT_SIZE']
    
    # Build the model using the trial's hyperparameters
    model = build_efficientnet_b0_from_trial(trial, input_shape=(input_height, input_width, 3), num_classes=10)

    # Directory for checkpoints
    study_dir = "experiments/EfficientNetB0/tuning"  # Adjust to your experiment folder
    checkpoint_dir = os.path.join(study_dir, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Define the checkpoint callback
    checkpoint_callback = ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, f"trial_{trial.number}.h5"),
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1
    )
    
    # Callbacks for early stopping, pruning, and model checkpointing
    callbacks = [
        EarlyStopping('val_loss', patience=5, restore_best_weights=True),
        TFKerasPruningCallback(trial, monitor="val_accuracy"),
        checkpoint_callback
    ]

    # Train the model
    history = model.fit(
        train_generator,  # Replace with your actual train generator
        validation_data=valid_generator,  # Replace with your actual validation generator
        epochs=30,
        callbacks=callbacks,
        verbose=1,
        workers=8,
        class_weight=new_class_weights,  # Use class weights if needed
    )

    # Get the best validation accuracy achieved during training
    val_acc = max(history.history['val_accuracy'])

    # Access the dropout layer and dense layers by their correct names
    dropout_layer = model.get_layer('dropout_1')  # 'dropout_1' is the explicit name
    dropout_rate = dropout_layer.rate if dropout_layer is not None else None

    dense_layer_1 = model.get_layer('dense_1')  # 'dense_1' is the explicit name
    dense_units_1 = dense_layer_1.units if dense_layer_1 is not None else None

    # Print trial results using the correct layer names
    print(f"[Trial {trial.number}] val_accuracy: {val_acc:.4f} | "
          f"lr: {model.optimizer.learning_rate.numpy():.5f} | "
          f"dropout: {dropout_rate:.2f} | "
          f"dense_1 units: {dense_units_1}")

    # Clear the session to free memory
    tf.keras.backend.clear_session()
    gc.collect()

    return val_acc


In [ ]:
# Optuna Study Setup
import os
import shutil
import json
import pandas as pd

# Define directories for saving the results
efficientnet_result_dir = os.path.join(CONFIG['MODEL_DIR'], "EfficientNetB0_Tuned")
os.makedirs(efficientnet_result_dir, exist_ok=True)

# Create the Optuna study
study_efficientnet = optuna.create_study(
    study_name="EfficientNetB0_Tuning",  # Study name
    direction="maximize",  # We want to maximize validation accuracy
    sampler=optuna.samplers.TPESampler(seed=CONFIG['SEED']),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)  # Pruning for faster optimization
)

# Run the optimization for a specified number of trials
study_efficientnet.optimize(objective_efficientnet_b0, n_trials=20)


[I 2025-05-15 01:15:58,090] A new study created in memory with name: EfficientNetB0_Tuning


Epoch 1/30
330/330 [==============================] - ETA: 0s - loss: 2.1542 - accuracy: 0.1307
Epoch 1: val_loss improved from inf to 2.10596, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_0.h5
330/330 [==============================] - 18s 51ms/step - loss: 2.1542 - accuracy: 0.1307 - val_loss: 2.1060 - val_accuracy: 0.0413
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 1.9926 - accuracy: 0.1595
Epoch 2: val_loss improved from 2.10596 to 2.05443, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_0.h5
330/330 [==============================] - 16s 48ms/step - loss: 1.9931 - accuracy: 0.1596 - val_loss: 2.0544 - val_accuracy: 0.1633
Epoch 3/30
330/330 [==============================] - ETA: 0s - loss: 1.8981 - accuracy: 0.1997
Epoch 3: val_loss did not improve from 2.05443
330/330 [==============================] - 16s 47ms/step - loss: 1.8981 - accuracy: 0.1997 - val_loss: 2.0873 - val_accuracy: 0.1292
Epoch 4/30
330/330 [

[I 2025-05-15 01:23:47,227] Trial 0 finished with value: 0.7065321803092957 and parameters: {'dropout_rate': 0.4123620356542087, 'dense_units': 512, 'lr': 0.0029106359131330704}. Best is trial 0 with value: 0.7065321803092957.


[Trial 0] val_accuracy: 0.7065 | lr: 0.00291 | dropout: 0.41 | dense_1 units: 512
Epoch 1/30
329/330 [============================>.] - ETA: 0s - loss: 2.2989 - accuracy: 0.1142
Epoch 1: val_loss improved from inf to 2.30724, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_1.h5
330/330 [==============================] - 16s 46ms/step - loss: 2.2980 - accuracy: 0.1140 - val_loss: 2.3072 - val_accuracy: 0.0235
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 2.2391 - accuracy: 0.1224
Epoch 2: val_loss improved from 2.30724 to 2.26379, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_1.h5
330/330 [==============================] - 15s 46ms/step - loss: 2.2394 - accuracy: 0.1225 - val_loss: 2.2638 - val_accuracy: 0.1061
Epoch 3/30
329/330 [============================>.] - ETA: 0s - loss: 2.1533 - accuracy: 0.1344
Epoch 3: val_loss improved from 2.26379 to 2.24526, saving model to experiments/EfficientNetB0/tuning\checkpoints\tria

[I 2025-05-15 01:31:38,832] Trial 1 finished with value: 0.20509125292301178 and parameters: {'dropout_rate': 0.47959754525911097, 'dense_units': 128, 'lr': 0.00020511104188433984}. Best is trial 0 with value: 0.7065321803092957.


...

[I 2025-05-15 02:28:54,224] Trial 17 pruned. Trial was pruned at epoch 6.


Epoch 1/30
330/330 [==============================] - ETA: 0s - loss: 2.1860 - accuracy: 0.1167
Epoch 1: val_loss improved from inf to 2.17910, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_18.h5
330/330 [==============================] - 16s 48ms/step - loss: 2.1860 - accuracy: 0.1167 - val_loss: 2.1791 - val_accuracy: 0.1023
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 2.0406 - accuracy: 0.1401
Epoch 2: val_loss improved from 2.17910 to 2.03113, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_18.h5
330/330 [==============================] - 16s 47ms/step - loss: 2.0408 - accuracy: 0.1401 - val_loss: 2.0311 - val_accuracy: 0.1119
Epoch 3/30
329/330 [============================>.] - ETA: 0s - loss: 1.9384 - accuracy: 0.1711
Epoch 3: val_loss did not improve from 2.03113
330/330 [==============================] - 16s 47ms/step - loss: 1.9390 - accuracy: 0.1710 - val_loss: 2.1886 - val_accuracy: 0.1138
Epoch 4/30
329/330

[I 2025-05-15 02:30:44,783] Trial 18 pruned. Trial was pruned at epoch 6.


Epoch 1/30
329/330 [============================>.] - ETA: 0s - loss: 2.1412 - accuracy: 0.1403
Epoch 1: val_loss improved from inf to 2.33072, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_19.h5
330/330 [==============================] - 16s 45ms/step - loss: 2.1405 - accuracy: 0.1401 - val_loss: 2.3307 - val_accuracy: 0.1105
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 1.8930 - accuracy: 0.2009
Epoch 2: val_loss improved from 2.33072 to 1.98434, saving model to experiments/EfficientNetB0/tuning\checkpoints\trial_19.h5
330/330 [==============================] - 15s 45ms/step - loss: 1.8923 - accuracy: 0.2011 - val_loss: 1.9843 - val_accuracy: 0.1647
Epoch 3/30
330/330 [==============================] - ETA: 0s - loss: 1.7009 - accuracy: 0.2573
Epoch 3: val_loss did not improve from 1.98434
330/330 [==============================] - 17s 51ms/step - loss: 1.7009 - accuracy: 0.2573 - val_loss: 2.5108 - val_accuracy: 0.1234
Epoch 4/30
329/330

[I 2025-05-15 02:35:20,737] Trial 19 finished with value: 0.6484149694442749 and parameters: {'dropout_rate': 0.3194684643948313, 'dense_units': 128, 'lr': 0.009444268410961857}. Best is trial 0 with value: 0.7065321803092957.


In [ ]:
# Save all trials' results to CSV
trials_df = study_efficientnet.trials_dataframe()
trials_df.to_csv(os.path.join(efficientnet_result_dir, "optuna_trials_efficientnet.csv"), index=False)

# Save best trial parameters and results to a CSV
best_trial = study_efficientnet.best_trial
best_result = {
    **best_trial.params,
    "best_value": best_trial.value,
    "trial_number": best_trial.number
}
pd.DataFrame([best_result]).to_csv(os.path.join(efficientnet_result_dir, "best_params_efficientnet.csv"), index=False)

# Save best model weights and parameters
best_dir = "experiments/EfficientNetB0/tuning"
os.makedirs(best_dir, exist_ok=True)

# Save the best hyperparameters to JSON
with open(f"{best_dir}/best_params.json", "w") as f:
    json.dump(study_efficientnet.best_params, f, indent=2)

# Get the checkpoint file from the best trial and copy it
checkpoint_path = f"experiments/EfficientNetB0/tuning/checkpoints/trial_{study_efficientnet.best_trial.number}.h5"
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, f"{best_dir}/best_model_efficientnet.h5")
else:
    print(f"Checkpoint file not found: {checkpoint_path}")

# Print final results
print(f"\n✅ Tuning complete. Best val_accuracy: {best_result['best_value']:.4f}")
print(f"📄 Saved to folder: {efficientnet_result_dir}")


✅ Tuning complete. Best val_accuracy: 0.7065
📄 Saved to folder: saved_models/EfficientNetB0_Tuned


### 4.3. ResNet18

##### 4.3.1. ResNet18 Definition

In [ ]:
# Define resnet18 architecture
def build_resnet18_from_trial(trial, input_shape, num_classes):
    def conv_bn_relu(x, filters, kernel_size=3, strides=1):
        x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        return layers.ReLU()(x)

    def residual_block(x, filters, strides=1, downsample=False):
        shortcut = x
        if downsample:
            shortcut = layers.Conv2D(filters, 1, strides=strides, padding='same', use_bias=False)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        
        x = conv_bn_relu(x, filters, strides=strides)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        
        x = layers.Add()([x, shortcut])
        return layers.ReLU()(x)

    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units = trial.suggest_int("dense_units", 128, 512)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)

    inputs = Input(shape=input_shape)
    x = conv_bn_relu(inputs, 64, kernel_size=7, strides=2)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

    for filters, reps, stride in zip([64, 128, 256, 512], [2, 2, 2, 2], [1, 2, 2, 2]):
        for i in range(reps):
            x = residual_block(x, filters, strides=stride if i == 0 else 1, downsample=(i == 0 and stride != 1))

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

##### 4.3.2. ResNet18 Optuna Tuning

In [27]:
# Objective function for Optuna
def objective_resnet18(trial):
    input_height, input_width = CONFIG['INPUT_SIZE']
    input_shape = (input_height, input_width, 3)
    num_classes = len(train_generator.class_indices)

    model = build_resnet18_from_trial(trial, input_shape, num_classes)

    #  Local generator fix: define correct target_size as a flat tuple
    fixed_offline_gen = ImageDataGenerator(rescale=1./255)
    
    # Replace offline_train_df with the actual DataFrame you're using
    fixed_train_gen = fixed_offline_gen.flow_from_dataframe(
        dataframe=oversampled_train_df,  
        directory=CONFIG['IMAGE_DIR_PROCESSED'],  
        x_col="filepath",
        y_col="variety",
        target_size=CONFIG['INPUT_SIZE'],  
        batch_size=CONFIG['BATCH_SIZE'],
        class_mode='categorical',
        shuffle=True,
        seed=CONFIG['SEED']
    )

    # Define callbacks
    study_dir = "experiments/ResNet18/tuning"
    checkpoint_dir = os.path.join(study_dir, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_path = os.path.join(checkpoint_dir, f"trial_{trial.number}.h5")
    checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1,
        workers=8
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True),
        TFKerasPruningCallback(trial, monitor="val_accuracy"),
        checkpoint_callback
    ]

    # Train the model
    history = model.fit(
        fixed_train_gen,
        validation_data=valid_generator,
        epochs=CONFIG['EPOCHS_TUNING'],
        callbacks=callbacks,
        verbose=1,
        class_weight=new_class_weights,
        workers=8
    )

    val_acc = max(history.history['val_accuracy'])
    tf.keras.backend.clear_session()
    gc.collect()
    return val_acc

In [ ]:
# Start the Optuna study
resnet_result_dir = os.path.join(CONFIG['MODEL_DIR'], "ResNet18_Tuned")
os.makedirs(resnet_result_dir, exist_ok=True)

study_resnet = optuna.create_study(
    study_name="ResNet18_Tuning",
    direction="maximize",
    sampler=TPESampler(seed=CONFIG['SEED']),
    pruner=MedianPruner(n_warmup_steps=5)
)

study_resnet.optimize(objective_resnet18, n_trials=CONFIG['OPTUNA_TRIALS'])

[I 2025-05-15 02:39:18,324] A new study created in memory with name: ResNet18_Tuning
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\3102420762.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Found 10546 validated image filenames belonging to 10 classes.
Epoch 1/30
329/330 [============================>.] - ETA: 0s - loss: 1.8633 - accuracy: 0.2119
Epoch 1: val_loss improved from inf to 4.06865, saving model to experiments/ResNet18/tuning\checkpoints\trial_0.h5
330/330 [==============================] - 21s 59ms/step - loss: 1.8628 - accuracy: 0.2121 - val_loss: 4.0687 - val_accuracy: 0.1595
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 1.3300 - accuracy: 0.2889
Epoch 2: val_loss improved from 4.06865 to 3.69142, saving model to experiments/ResNet18/tuning\checkpoints\trial_0.h5
330/330 [==============================] - 18s 53ms/step - loss: 1.3284 - accuracy: 0.2888 - val_loss: 3.6914 - val_accuracy: 0.1407
Epoch 3/30
330/330 [==============================] - ETA: 0s - loss: 1.0679 - accuracy: 0.3501
Epoch 3: val_loss did not improve from 3.69142
330/330 [==============================] - 17s 53ms/step - loss: 1.0679 - accuracy: 0.3501 - val_loss:

[I 2025-05-15 02:46:23,931] Trial 0 finished with value: 0.878962516784668 and parameters: {'dropout_rate': 0.3123620356542087, 'dense_units': 494, 'lr': 0.0029106359131330704}. Best is trial 0 with value: 0.878962516784668.
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\3102420762.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Found 10546 validated image filenames belonging to 10 classes.
Epoch 1/30
330/330 [==============================] - ETA: 0s - loss: 1.0458 - accuracy: 0.4699
Epoch 1: val_loss improved from inf to 7.03467, saving model to experiments/ResNet18/tuning\checkpoints\trial_1.h5
330/330 [==============================] - 20s 58ms/step - loss: 1.0458 - accuracy: 0.4699 - val_loss: 7.0347 - val_accuracy: 0.0533
Epoch 2/30
329/330 [============================>.] - ETA: 0s - loss: 0.4336 - accuracy: 0.6869
Epoch 2: val_loss improved from 7.03467 to 1.21970, saving model to experiments/ResNet18/tuning\checkpoints\trial_1.h5
330/330 [==============================] - 18s 53ms/step - loss: 0.4330 - accuracy: 0.6870 - val_loss: 1.2197 - val_accuracy: 0.5961
Epoch 3/30
330/330 [==============================] - ETA: 0s - loss: 0.2536 - accuracy: 0.7857
Epoch 3: val_loss improved from 1.21970 to 0.82336, saving model to experiments/ResNet18/tuning\checkpoints\trial_1.h5
330/330 [=====================

[I 2025-05-15 02:50:49,291] Trial 1 finished with value: 0.9058597683906555 and parameters: {'dropout_rate': 0.379597545259111, 'dense_units': 188, 'lr': 0.00020511104188433984}. Best is trial 1 with value: 0.9058597683906555.
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\3102420762.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


...

[I 2025-05-15 03:57:35,790] Trial 17 pruned. Trial was pruned at epoch 7.
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\3102420762.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Found 10546 validated image filenames belonging to 10 classes.
Epoch 1/30
329/330 [============================>.] - ETA: 0s - loss: 1.0094 - accuracy: 0.4919
Epoch 1: val_loss improved from inf to 3.73267, saving model to experiments/ResNet18/tuning\checkpoints\trial_18.h5
330/330 [==============================] - 20s 58ms/step - loss: 1.0081 - accuracy: 0.4923 - val_loss: 3.7327 - val_accuracy: 0.0485
Epoch 2/30
330/330 [==============================] - ETA: 0s - loss: 0.3604 - accuracy: 0.7161
Epoch 2: val_loss improved from 3.73267 to 2.52050, saving model to experiments/ResNet18/tuning\checkpoints\trial_18.h5
330/330 [==============================] - 19s 57ms/step - loss: 0.3604 - accuracy: 0.7161 - val_loss: 2.5205 - val_accuracy: 0.2992
Epoch 3/30
329/330 [============================>.] - ETA: 0s - loss: 0.2244 - accuracy: 0.8006
Epoch 3: val_loss improved from 2.52050 to 2.43805, saving model to experiments/ResNet18/tuning\checkpoints\trial_18.h5
330/330 [==================

[I 2025-05-15 04:01:43,896] Trial 18 finished with value: 0.9125840663909912 and parameters: {'dropout_rate': 0.43157720371402136, 'dense_units': 508, 'lr': 0.00018252167986211796}. Best is trial 8 with value: 0.9476464986801147.
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5248\3102420762.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Found 10546 validated image filenames belonging to 10 classes.
Epoch 1/30
329/330 [============================>.] - ETA: 0s - loss: 0.9445 - accuracy: 0.5010
Epoch 1: val_loss improved from inf to 5.14786, saving model to experiments/ResNet18/tuning\checkpoints\trial_19.h5
330/330 [==============================] - 20s 59ms/step - loss: 0.9428 - accuracy: 0.5015 - val_loss: 5.1479 - val_accuracy: 0.0461
Epoch 2/30
330/330 [==============================] - ETA: 0s - loss: 0.3555 - accuracy: 0.7133
Epoch 2: val_loss improved from 5.14786 to 0.93971, saving model to experiments/ResNet18/tuning\checkpoints\trial_19.h5
330/330 [==============================] - 19s 58ms/step - loss: 0.3555 - accuracy: 0.7133 - val_loss: 0.9397 - val_accuracy: 0.7267
Epoch 3/30
329/330 [============================>.] - ETA: 0s - loss: 0.1934 - accuracy: 0.8297
Epoch 3: val_loss did not improve from 0.93971
330/330 [==============================] - 19s 57ms/step - loss: 0.1932 - accuracy: 0.8295 - val_los

[I 2025-05-15 04:04:36,636] Trial 19 finished with value: 0.8342939615249634 and parameters: {'dropout_rate': 0.4135933406640236, 'dense_units': 509, 'lr': 0.00015418160822557324}. Best is trial 8 with value: 0.9476464986801147.


In [ ]:
# Save all trial data
trials_df = study_resnet.trials_dataframe()
trials_df.to_csv(os.path.join(resnet_result_dir, "optuna_trials_resnet18.csv"), index=False)

# Save best trial info
best_trial = study_resnet.best_trial
best_result = {
    **best_trial.params,
    "best_value": best_trial.value,
    "trial_number": best_trial.number
}
pd.DataFrame([best_result]).to_csv(os.path.join(resnet_result_dir, "best_params_resnet18.csv"), index=False)

# Save best model weights
best_dir = "experiments/ResNet18/tuning"
os.makedirs(best_dir, exist_ok=True)

with open(os.path.join(best_dir, "best_params.json"), "w") as f:
    json.dump(study_resnet.best_params, f, indent=2)

checkpoint_path = os.path.join(best_dir, "checkpoints", f"trial_{best_result['trial_number']}.h5")
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, os.path.join(best_dir, "best_model_resnet18.h5"))
else:
    print(f"Checkpoint not found: {checkpoint_path}")

print(f"\n✅ ResNet18 tuning complete. Best val_accuracy: {best_result['best_value']:.4f}")
print(f"📄 Results saved to: {resnet_result_dir}")


✅ ResNet18 tuning complete. Best val_accuracy: 0.9476
📄 Results saved to: saved_models/ResNet18_Tuned


### 4.4. MobileNetv2-Lite

##### 4.4.1. MobileNetV2-Lite Definition

In [ ]:
# Define the definition of mobilenetv2 architecture
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam

def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

def se_block(inputs, se_ratio=0.25):
    filters = inputs.shape[-1]
    reduced_filters = max(1, int(filters * se_ratio))
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Reshape((1, 1, filters))(x)
    x = layers.Conv2D(reduced_filters, 1, activation='relu')(x)
    x = layers.Conv2D(filters, 1, activation='hard_sigmoid')(x)
    return layers.Multiply()([inputs, x])

def mbv2_block(inputs, out_channels, expansion, stride, se, activation, block_id):
    in_channels = inputs.shape[-1]
    x = inputs

    if expansion != 1:
        x = layers.Conv2D(in_channels * expansion, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    if se:
        x = se_block(x)

    x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and in_channels == out_channels:
        x = layers.Add()([inputs, x])

    return x

def build_mobilenetv2_lite_from_trial(trial, input_shape, num_classes):
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units = trial.suggest_int("dense_units", 128, 512)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)

    activation = hard_swish
    inputs = layers.Input(shape=input_shape, name="input")

    # Initial Conv layer
    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # MobileNetV2 blocks with fewer filters and reduced depth (for efficiency)
    x = mbv2_block(x, 16, 1, 2, True, activation, 0)
    x = mbv2_block(x, 24, 4, 2, False, activation, 1)
    x = mbv2_block(x, 24, 3, 1, False, activation, 2)
    x = mbv2_block(x, 40, 4, 2, True, activation, 3)
    x = mbv2_block(x, 40, 6, 1, True, activation, 4)

    # Global average pooling layer
    x = layers.GlobalAveragePooling2D()(x)

    # Fully connected layer
    x = layers.Dense(dense_units, activation=activation)(x)
    x = layers.Dropout(dropout_rate)(x)

    # Output layer
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    # Create and compile the model
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

##### 4.4.2. MobileNetV2-Lite Optuna Tuning

In [ ]:
# Retraining the Model and Saving the Best Trial

def objective_mbv2_lite(trial):
    input_shape = CONFIG['INPUT_SIZE'] + (3,)  # Adding channel dimension for RGB images
    num_classes = len(train_generator.class_indices)  # Number of classes from the generator

    # Build the model with trial-based hyperparameters
    model = build_mobilenetv2_lite_from_trial(trial, input_shape, num_classes)

    # Directory for saving checkpoints during training
    study_dir = "experiments/MobileNetV2_Lite/tuning"
    checkpoint_dir = os.path.join(study_dir, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Set up ModelCheckpoint to save the best model during training
    checkpoint_path = os.path.join(checkpoint_dir, f"trial_{trial.number}.h5")
    checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=0
    )

    # Callbacks for early stopping and pruning (Optuna integration)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True),
        TFKerasPruningCallback(trial, monitor="val_accuracy"),
        checkpoint_callback
    ]

    # Train the model
    history = model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=CONFIG['EPOCHS_TUNING'],
        callbacks=callbacks,
        verbose=1,
        class_weight=new_class_weights,  # Apply class weights to deal with imbalance
        workers=8
    )

    # Get the best validation accuracy achieved during training
    val_acc = max(history.history['val_accuracy'])

    # Free up memory after the trial
    tf.keras.backend.clear_session()
    gc.collect()

    return val_acc


In [ ]:
# Optuna study for tuning MobileNetV2 Lite
mbv2_lite_result_dir = os.path.join(CONFIG['MODEL_DIR'], "MobileNetV2_Lite_Tuned")
os.makedirs(mbv2_lite_result_dir, exist_ok=True)

study_mbv2_lite = optuna.create_study(
    study_name="MobileNetV2_Lite_Tuning",
    direction="maximize",
    sampler=TPESampler(seed=CONFIG['SEED']),
    pruner=MedianPruner(n_warmup_steps=5)
)

study_mbv2_lite.optimize(objective_mbv2_lite, n_trials=CONFIG['OPTUNA_TRIALS'])


[I 2025-05-15 05:06:44,467] A new study created in memory with name: MobileNetV2_Lite_Tuning
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18328\3477773601.py:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)


Epoch 1/30
330/330 [==============================] - 16s 37ms/step - loss: 1.4427 - accuracy: 0.3159 - val_loss: 3.6083 - val_accuracy: 0.0360
Epoch 2/30
330/330 [==============================] - 12s 36ms/step - loss: 0.9743 - accuracy: 0.4771 - val_loss: 7.8397 - val_accuracy: 0.0476
Epoch 3/30
330/330 [==============================] - 12s 36ms/step - loss: 0.7109 - accuracy: 0.5774 - val_loss: 3.8408 - val_accuracy: 0.2142
Epoch 4/30
330/330 [==============================] - 12s 37ms/step - loss: 0.5372 - accuracy: 0.6567 - val_loss: 1.5522 - val_accuracy: 0.4563
Epoch 5/30
330/330 [==============================] - 12s 36ms/step - loss: 0.4497 - accuracy: 0.6965 - val_loss: 1.2875 - val_accuracy: 0.5965
Epoch 6/30
330/330 [==============================] - 12s 37ms/step - loss: 0.3996 - accuracy: 0.7286 - val_loss: 1.1556 - val_accuracy: 0.6739
Epoch 7/30
330/330 [==============================] - 12s 37ms/step - loss: 0.3584 - accuracy: 0.7500 - val_loss: 1.0467 - val_accuracy:

[I 2025-05-15 05:09:16,743] Trial 0 finished with value: 0.6834774017333984 and parameters: {'dropout_rate': 0.3123620356542087, 'dense_units': 494, 'lr': 0.0029106359131330704}. Best is trial 0 with value: 0.6834774017333984.


Epoch 1/30
330/330 [==============================] - 14s 38ms/step - loss: 1.7587 - accuracy: 0.2144 - val_loss: 2.4800 - val_accuracy: 0.0360
Epoch 2/30
330/330 [==============================] - 12s 36ms/step - loss: 1.2739 - accuracy: 0.3201 - val_loss: 2.0956 - val_accuracy: 0.2104
Epoch 3/30
330/330 [==============================] - 12s 36ms/step - loss: 1.0541 - accuracy: 0.3982 - val_loss: 1.9038 - val_accuracy: 0.3074
Epoch 4/30
330/330 [==============================] - 12s 37ms/step - loss: 0.8829 - accuracy: 0.4768 - val_loss: 1.6913 - val_accuracy: 0.4155
Epoch 5/30
330/330 [==============================] - 12s 36ms/step - loss: 0.7606 - accuracy: 0.5440 - val_loss: 1.4602 - val_accuracy: 0.4443
Epoch 6/30
330/330 [==============================] - 12s 36ms/step - loss: 0.6807 - accuracy: 0.5905 - val_loss: 0.9985 - val_accuracy: 0.6302
Epoch 7/30
330/330 [==============================] - 12s 36ms/step - loss: 0.5908 - accuracy: 0.6234 - val_loss: 1.2190 - val_accuracy:

[I 2025-05-15 05:15:26,042] Trial 1 finished with value: 0.7958693504333496 and parameters: {'dropout_rate': 0.379597545259111, 'dense_units': 188, 'lr': 0.00020511104188433984}. Best is trial 1 with value: 0.7958693504333496.


...

[I 2025-05-15 06:12:48,209] Trial 17 pruned. Trial was pruned at epoch 6.


Epoch 1/30
330/330 [==============================] - 15s 40ms/step - loss: 1.5940 - accuracy: 0.2509 - val_loss: 2.0180 - val_accuracy: 0.0360
Epoch 2/30
330/330 [==============================] - 13s 39ms/step - loss: 1.0845 - accuracy: 0.3877 - val_loss: 1.3401 - val_accuracy: 0.4707
Epoch 3/30
330/330 [==============================] - 13s 38ms/step - loss: 0.8926 - accuracy: 0.4686 - val_loss: 2.2914 - val_accuracy: 0.2560
Epoch 4/30
330/330 [==============================] - 13s 39ms/step - loss: 0.7421 - accuracy: 0.5404 - val_loss: 1.8326 - val_accuracy: 0.3650
Epoch 5/30
330/330 [==============================] - 13s 39ms/step - loss: 0.6117 - accuracy: 0.5971 - val_loss: 1.9916 - val_accuracy: 0.3996
Epoch 6/30
330/330 [==============================] - ETA: 0s - loss: 0.5312 - accuracy: 0.6453

[I 2025-05-15 06:14:08,696] Trial 18 pruned. Trial was pruned at epoch 5.


Epoch 1/30
330/330 [==============================] - 15s 40ms/step - loss: 1.4907 - accuracy: 0.2830 - val_loss: 2.2481 - val_accuracy: 0.0490
Epoch 2/30
330/330 [==============================] - 13s 38ms/step - loss: 0.9456 - accuracy: 0.4702 - val_loss: 3.5261 - val_accuracy: 0.2656
Epoch 3/30
330/330 [==============================] - 13s 39ms/step - loss: 0.6671 - accuracy: 0.5961 - val_loss: 2.3710 - val_accuracy: 0.3554
Epoch 4/30
330/330 [==============================] - 13s 39ms/step - loss: 0.5164 - accuracy: 0.6579 - val_loss: 2.2970 - val_accuracy: 0.3732
Epoch 5/30
330/330 [==============================] - 13s 39ms/step - loss: 0.4256 - accuracy: 0.7005 - val_loss: 2.1921 - val_accuracy: 0.3314
Epoch 6/30
330/330 [==============================] - ETA: 0s - loss: 0.3424 - accuracy: 0.7452

[I 2025-05-15 06:15:29,787] Trial 19 pruned. Trial was pruned at epoch 5.


In [ ]:
# Save the trial results
trials_df = study_mbv2_lite.trials_dataframe()
trials_df.to_csv(os.path.join(mbv2_lite_result_dir, "optuna_trials_mbv2_lite.csv"), index=False)

# Save the best trial parameters and results
best_trial = study_mbv2_lite.best_trial
best_result = {
    **best_trial.params,
    "best_value": best_trial.value,
    "trial_number": best_trial.number
}
pd.DataFrame([best_result]).to_csv(os.path.join(mbv2_lite_result_dir, "best_params_mbv2_lite.csv"), index=False)

# Save the best model weights
best_dir = "experiments/MobileNetV2_Lite/tuning"
os.makedirs(best_dir, exist_ok=True)

with open(os.path.join(best_dir, "best_params.json"), "w") as f:
    json.dump(study_mbv2_lite.best_params, f, indent=2)

checkpoint_path = os.path.join(best_dir, "checkpoints", f"trial_{best_result['trial_number']}.h5")
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, os.path.join(best_dir, "best_model_mbv2_lite.h5"))
else:
    print(f"Checkpoint not found: {checkpoint_path}")

print(f"\n✅ Tuning complete. Best val_accuracy: {best_result['best_value']:.4f}")
print(f"📄 Saved to folder: {mbv2_lite_result_dir}")


✅ Tuning complete. Best val_accuracy: 0.8852
📄 Saved to folder: saved_models/MobileNetV2_Lite_Tuned


## 5.Retrain

### 5.1. EfficientNetB0

In [13]:
# Load the best parameters for EfficientNetB0 from the correct CSV file path
efficientnet_params = pd.read_csv('saved_models/EfficientNetB0_Tuned/best_params_efficientnet.csv')
best_params_efficientnet = efficientnet_params.iloc[0]
dropout_rate = best_params_efficientnet['dropout_rate']
dense_units = best_params_efficientnet['dense_units']
lr = best_params_efficientnet['lr']

print(f"Best parameters for EfficientNetB0:\n Dropout Rate: {dropout_rate}, Dense Units: {dense_units}, Learning Rate: {lr}")


Best parameters for EfficientNetB0:
 Dropout Rate: 0.4123620356542087, Dense Units: 512.0, Learning Rate: 0.002910635913133


In [14]:
# Rebuild EfficientNetB0 model with the best parameters
def build_efficientnet_b0_from_params(best_params, input_shape, num_classes):
    dropout_rate = best_params['dropout_rate']
    dense_units = best_params['dense_units']
    lr = best_params['lr']

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, strides=2, padding='same', activation='swish')(inputs)
    
    # Add more layers here as per EfficientNetB0 architecture
    for i in range(3):
        x_res = x
        x = layers.DepthwiseConv2D(3, padding='same', activation='swish', name=f'depthwise_conv2d_{i}')(x)
        x = layers.Conv2D(32, 1, activation='swish', name=f'conv2d_{i}')(x)
        if x.shape == x_res.shape:
            x = layers.Add(name=f'add_{i}')([x, x_res])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='swish')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.models.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])

    return model

# Build the EfficientNetB0 model using the best hyperparameters
model = build_efficientnet_b0_from_params(best_params_efficientnet, (CONFIG['INPUT_SIZE'][0], CONFIG['INPUT_SIZE'][1], 3), len(train_generator.class_indices))

In [ ]:
# Define callbacks
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(CONFIG['MODEL_DIR'], 'checkpoints', "best_model.h5"),
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['EARLY_STOP_PATIENCE'],
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# Train the model
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=CONFIG['EPOCHS'],
    callbacks=[checkpoint_callback, early_stopping, reduce_lr],
    verbose=1,
    class_weight=new_class_weights,  # Use class weights if needed
    workers=8
)

# Save the retrained model
model.save(os.path.join(CONFIG['MODEL_DIR'], 'checkpoints', "efficientnetb0_retrained.h5"))
print("EfficientNetB0 retraining complete.")

Epoch 1/100
260/261 [============================>.] - ETA: 0s - loss: 1.3481 - accuracy: 0.2864
Epoch 1: val_loss improved from inf to 1.79654, saving model to saved_models/checkpoints\best_model.h5
261/261 [==============================] - 15s 55ms/step - loss: 1.3462 - accuracy: 0.2864 - val_loss: 1.7965 - val_accuracy: 0.2334 - lr: 0.0029
Epoch 2/100
261/261 [==============================] - ETA: 0s - loss: 1.2857 - accuracy: 0.3447
Epoch 2: val_loss improved from 1.79654 to 1.72207, saving model to saved_models/checkpoints\best_model.h5
261/261 [==============================] - 14s 53ms/step - loss: 1.2857 - accuracy: 0.3447 - val_loss: 1.7221 - val_accuracy: 0.3900 - lr: 0.0029
Epoch 3/100
261/261 [==============================] - ETA: 0s - loss: 1.2447 - accuracy: 0.3730
Epoch 3: val_loss improved from 1.72207 to 1.65261, saving model to saved_models/checkpoints\best_model.h5
261/261 [==============================] - 14s 54ms/step - loss: 1.2447 - accuracy: 0.3730 - val_los

### 5.2. MobileNetV3-Small

In [15]:
# Load the best parameters for MobileNetV3-Small
mbv3_params = pd.read_csv('saved_models/MobileNetV3-Small_Tuned/best_params_mbv3.csv')
best_params_mbv3 = mbv3_params.iloc[0]
dropout_rate = best_params_mbv3['dropout_rate']
dense_units = best_params_mbv3['dense_units']
lr = best_params_mbv3['lr']

print(f"Best parameters for MobileNetV3-Small:\n Dropout Rate: {dropout_rate}, Dense Units: {dense_units}, Learning Rate: {lr}")

Best parameters for MobileNetV3-Small:
 Dropout Rate: 0.2856152795491966, Dense Units: 188.0, Learning Rate: 0.0004839034242502


In [16]:
# Define the hard_swish function
def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

# Define the SE block (Squeeze-and-Excitation Block)
def se_block(inputs, se_ratio=0.25):
    filters = inputs.shape[-1]
    reduced_filters = max(1, int(filters * se_ratio))
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Reshape((1, 1, filters))(x)
    x = layers.Conv2D(reduced_filters, 1, activation='relu')(x)
    x = layers.Conv2D(filters, 1, activation='hard_sigmoid')(x)
    return layers.Multiply()([inputs, x])

# Define the MobileNetV3 block
def mbv3_block(inputs, out_channels, expansion, stride, se, activation, block_id):
    in_channels = inputs.shape[-1]
    x = inputs

    if expansion != 1:
        x = layers.Conv2D(in_channels * expansion, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    if se:
        x = se_block(x)

    x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and in_channels == out_channels:
        x = layers.Add()([inputs, x])

    return x

# Rebuild MobileNetV3-Small model with the best parameters
def build_mobilenetv3_small_from_params(best_params, input_shape, num_classes):
    dropout_rate = best_params['dropout_rate']
    dense_units = best_params['dense_units']
    lr = best_params['lr']

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Add MobileNetV3-Small blocks here
    x = mbv3_block(x, 16, 1, 2, True, 'relu', 0)  # Block 1
    x = mbv3_block(x, 24, 4, 2, False, 'relu', 1)  # Block 2
    x = mbv3_block(x, 24, 3, 1, False, 'relu', 2)  # Block 3
    x = mbv3_block(x, 40, 4, 2, True, hard_swish, 3)  # Block 4
    x = mbv3_block(x, 40, 6, 1, True, hard_swish, 4)  # Block 5
    x = mbv3_block(x, 48, 6, 1, True, hard_swish, 5)  # Block 6
    x = mbv3_block(x, 96, 6, 2, True, hard_swish, 6)  # Block 7
    x = mbv3_block(x, 96, 6, 1, True, hard_swish, 7)  # Block 8

    x = layers.Conv2D(576, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(hard_swish)(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Add the dense and dropout layers
    x = layers.Dense(dense_units, activation=hard_swish)(x)
    x = layers.Dropout(dropout_rate)(x)

    # Output layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.models.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Build the MobileNetV3-Small model using the best hyperparameters
mbv3_model = build_mobilenetv3_small_from_params(best_params_mbv3, (CONFIG['INPUT_SIZE'][0], CONFIG['INPUT_SIZE'][1], 3), len(train_generator.class_indices))

In [ ]:
# Define callbacks
mbv3_checkpoint_dir = os.path.join(CONFIG['MODEL_DIR'], 'MobileNetV3-Small_Tuned/checkpoints')
os.makedirs(mbv3_checkpoint_dir, exist_ok=True)

checkpoint_callback_mbv3 = ModelCheckpoint(
    filepath=os.path.join(mbv3_checkpoint_dir, "best_model.h5"),
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)

early_stopping_mbv3 = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['EARLY_STOP_PATIENCE'],
    restore_best_weights=True
)

# Retrain MobileNetV3-Small model
history_mbv3 = mbv3_model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=CONFIG['EPOCHS'],
    callbacks=[checkpoint_callback_mbv3, early_stopping_mbv3],
    verbose=1,
    workers=8,
    class_weight=new_class_weights  # Use class weights if needed
)

# Save the retrained model
mbv3_model.save(os.path.join(mbv3_checkpoint_dir, "mobilenetv3_small_retrained.h5"))
print("MobileNetV3-Small retraining complete.")

Epoch 1/100
260/261 [============================>.] - ETA: 0s - loss: 1.1338 - accuracy: 0.3722
Epoch 1: val_loss improved from inf to 1.99968, saving model to saved_models/MobileNetV3-Small_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 14s 44ms/step - loss: 1.1330 - accuracy: 0.3719 - val_loss: 1.9997 - val_accuracy: 0.0634
Epoch 2/100
261/261 [==============================] - ETA: 0s - loss: 0.9000 - accuracy: 0.4669
Epoch 2: val_loss did not improve from 1.99968
261/261 [==============================] - 11s 42ms/step - loss: 0.9000 - accuracy: 0.4669 - val_loss: 2.4966 - val_accuracy: 0.0615
Epoch 3/100
261/261 [==============================] - ETA: 0s - loss: 0.7513 - accuracy: 0.5409
Epoch 3: val_loss improved from 1.99968 to 0.98531, saving model to saved_models/MobileNetV3-Small_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 11s 40ms/step - loss: 0.7513 - accuracy: 0.5409 - val_loss: 0.9853 - val_accuracy: 0.6417
Epoch

### 5.3. ResNet18

In [17]:
# Load the best parameters for ResNet18
resnet_params = pd.read_csv('saved_models/ResNet18_Tuned/best_params_resnet18.csv')
best_params_resnet = resnet_params.iloc[0]
dropout_rate = best_params_resnet['dropout_rate']
dense_units = best_params_resnet['dense_units']
lr = best_params_resnet['lr']

print(f"Best parameters for ResNet18:\n Dropout Rate: {dropout_rate}, Dense Units: {dense_units}, Learning Rate: {lr}")


Best parameters for ResNet18:
 Dropout Rate: 0.3368209952651108, Dense Units: 430.0, Learning Rate: 0.0002508115686045


In [18]:
# Define the residual block used in ResNet18
def residual_block(x, filters, strides=1, downsample=False):
    shortcut = x
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # First convolution block
    x = layers.Conv2D(filters, 3, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Second convolution block
    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    if downsample:
        x = layers.Add()([x, shortcut])  # Add the residual shortcut connection
    return layers.ReLU()(x)

# Rebuild ResNet18 model with the best parameters
def build_resnet18_from_params(best_params, input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Residual blocks for ResNet18 (simplified)
    for filters, reps, stride in zip([64, 128, 256, 512], [2, 2, 2, 2], [1, 2, 2, 2]):
        for i in range(reps):
            x = residual_block(x, filters, strides=stride if i == 0 else 1, downsample=(i == 0 and stride != 1))

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(best_params['dense_units'], activation='relu')(x)
    x = layers.Dropout(best_params['dropout_rate'])(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.models.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=best_params['lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Build the ResNet18 model using the best hyperparameters
resnet_model = build_resnet18_from_params(best_params_resnet, (CONFIG['INPUT_SIZE'][0], CONFIG['INPUT_SIZE'][1], 3), len(train_generator.class_indices))

In [ ]:
# Define callbacks
resnet_checkpoint_dir = os.path.join(CONFIG['MODEL_DIR'], 'ResNet18_Tuned/checkpoints')
os.makedirs(resnet_checkpoint_dir, exist_ok=True)
checkpoint_callback_resnet = ModelCheckpoint(
    filepath=os.path.join(resnet_checkpoint_dir, "best_model.h5"),
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)
early_stopping_resnet = EarlyStopping(monitor='val_loss', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True)

# Retrain ResNet18
history_resnet = resnet_model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=CONFIG['EPOCHS'],
    callbacks=[checkpoint_callback_resnet, early_stopping_resnet],
    verbose=1,
    workers=8,
    class_weight=new_class_weights  # Use class weights if needed
)

# Save the retrained model
resnet_model.save(os.path.join(resnet_checkpoint_dir, "resnet18_retrained.h5"))
print("ResNet18 retraining complete.")

Epoch 1/100
261/261 [==============================] - ETA: 0s - loss: 0.2452 - accuracy: 0.8008
Epoch 1: val_loss improved from inf to 1.03471, saving model to saved_models/ResNet18_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 44s 169ms/step - loss: 0.2452 - accuracy: 0.8008 - val_loss: 1.0347 - val_accuracy: 0.6652
Epoch 2/100
261/261 [==============================] - ETA: 0s - loss: 0.2387 - accuracy: 0.8056
Epoch 2: val_loss did not improve from 1.03471
261/261 [==============================] - 43s 162ms/step - loss: 0.2387 - accuracy: 0.8056 - val_loss: 2.1280 - val_accuracy: 0.4918
Epoch 3/100
261/261 [==============================] - ETA: 0s - loss: 0.2279 - accuracy: 0.8119
Epoch 3: val_loss improved from 1.03471 to 1.02770, saving model to saved_models/ResNet18_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 43s 163ms/step - loss: 0.2279 - accuracy: 0.8119 - val_loss: 1.0277 - val_accuracy: 0.6902
Epoch 4/100
261/261 

### 5.4. MobileNetV2_Lite

In [19]:
# Load the best parameters for MobileNetV2_Lite from the tuning phase
mbv2_lite_params = pd.read_csv('saved_models/MobileNetV2_Lite_Tuned/best_params_mbv2_lite.csv')
best_params_mbv2_lite = mbv2_lite_params.iloc[0]
dropout_rate = best_params_mbv2_lite['dropout_rate']
dense_units = best_params_mbv2_lite['dense_units']
lr = best_params_mbv2_lite['lr']

print(f"Best parameters for MobileNetV2_Lite:\n Dropout Rate: {dropout_rate}, Dense Units: {dense_units}, Learning Rate: {lr}")


Best parameters for MobileNetV2_Lite:
 Dropout Rate: 0.2369365191800482, Dense Units: 295.0, Learning Rate: 0.0019288593267172


In [20]:
# Define hard_swish function for MobileNetV2_Lite
def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

# Define SE block (Squeeze-and-Excitation)
def se_block(inputs, se_ratio=0.25):
    filters = inputs.shape[-1]
    reduced_filters = max(1, int(filters * se_ratio))
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Reshape((1, 1, filters))(x)
    x = layers.Conv2D(reduced_filters, 1, activation='relu')(x)
    x = layers.Conv2D(filters, 1, activation='hard_sigmoid')(x)
    return layers.Multiply()([inputs, x])

# Define MobileNetV2 block
def mbv2_block(inputs, out_channels, expansion, stride, se, activation, block_id):
    in_channels = inputs.shape[-1]
    x = inputs

    if expansion != 1:
        x = layers.Conv2D(in_channels * expansion, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    if se:
        x = se_block(x)

    x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and in_channels == out_channels:
        x = layers.Add()([inputs, x])

    return x

# Rebuild MobileNetV2_Lite model with the best parameters
def build_mobilenetv2_lite_from_params(best_params, input_shape, num_classes):
    dropout_rate = best_params['dropout_rate']
    dense_units = best_params['dense_units']
    lr = best_params['lr']

    activation = hard_swish
    inputs = layers.Input(shape=input_shape, name="input")

    # Initial Conv layer
    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # MobileNetV2 blocks with fewer filters and reduced depth (for efficiency)
    x = mbv2_block(x, 16, 1, 2, True, activation, 0)
    x = mbv2_block(x, 24, 4, 2, False, activation, 1)
    x = mbv2_block(x, 24, 3, 1, False, activation, 2)
    x = mbv2_block(x, 40, 4, 2, True, activation, 3)
    x = mbv2_block(x, 40, 6, 1, True, activation, 4)

    # Global average pooling layer
    x = layers.GlobalAveragePooling2D()(x)

    # Fully connected layer
    x = layers.Dense(dense_units, activation=activation)(x)
    x = layers.Dropout(dropout_rate)(x)

    # Output layer
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    # Create and compile the model
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

In [ ]:
# Define callbacks
mbv2_lite_checkpoint_dir = os.path.join(CONFIG['MODEL_DIR'], 'MobileNetV2_Lite_Tuned/checkpoints')
os.makedirs(mbv2_lite_checkpoint_dir, exist_ok=True)
checkpoint_callback_mbv2_lite = ModelCheckpoint(
    filepath=os.path.join(mbv2_lite_checkpoint_dir, "best_model.h5"),
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)
early_stopping_mbv2_lite = EarlyStopping(monitor='val_loss', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True)

# Retrain MobileNetV2_Lite model
history_mbv2_lite = mbv2_lite_model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=CONFIG['EPOCHS'],
    callbacks=[checkpoint_callback_mbv2_lite, early_stopping_mbv2_lite],
    verbose=1,
    workers=8,
    class_weight=new_class_weights  # Use class weights if needed
)

# Save the retrained model
mbv2_lite_model.save(os.path.join(mbv2_lite_checkpoint_dir, "mobilenetv2_lite_retrained.h5"))
print("MobileNetV2_Lite retraining complete.")

Epoch 1/100
258/261 [============================>.] - ETA: 0s - loss: 0.8387 - accuracy: 0.5001
Epoch 1: val_loss improved from inf to 1.38733, saving model to saved_models/MobileNetV2_Lite_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 11s 41ms/step - loss: 0.8405 - accuracy: 0.4989 - val_loss: 1.3873 - val_accuracy: 0.4621
Epoch 2/100
258/261 [============================>.] - ETA: 0s - loss: 0.8320 - accuracy: 0.5025
Epoch 2: val_loss improved from 1.38733 to 1.30031, saving model to saved_models/MobileNetV2_Lite_Tuned/checkpoints\best_model.h5
261/261 [==============================] - 10s 38ms/step - loss: 0.8298 - accuracy: 0.5032 - val_loss: 1.3003 - val_accuracy: 0.4822
Epoch 3/100
258/261 [============================>.] - ETA: 0s - loss: 0.8275 - accuracy: 0.5030
Epoch 3: val_loss did not improve from 1.30031
261/261 [==============================] - 10s 37ms/step - loss: 0.8265 - accuracy: 0.5038 - val_loss: 1.7462 - val_accuracy: 0.3439
Epoch 4

## 6. Plotting models

In [ ]:
# Function to create directories for each model
def create_directories(base_dir, models):
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    for model in models:
        model_dir = os.path.join(base_dir, model)
        if not os.path.exists(model_dir):
            os.makedirs(model_dir)

# Function to save accuracy and loss plots for each model
def save_plots(history, model_name, base_dir):
    # Create subfolder for the model if not already present
    model_dir = os.path.join(base_dir, model_name)
    
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)
    
    # Save Accuracy Plot
    plt.plot(history.history['accuracy'], label='accuracy')
    plt.plot(history.history['val_accuracy'], label='val_accuracy')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='upper left')
    accuracy_plot_path = os.path.join(model_dir, f'{model_name}_accuracy.png')
    plt.savefig(accuracy_plot_path)
    plt.close()  # Close the plot to free up memory

    # Save Loss Plot
    plt.plot(history.history['loss'], label='loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper left')
    loss_plot_path = os.path.join(model_dir, f'{model_name}_loss.png')
    plt.savefig(loss_plot_path)
    plt.close()  # Close the plot to free up memory
    
    print(f"Plots saved to {model_dir}")

# Directory to store all plots
base_dir = 'model_plots'

# List of models
models = ['EfficientNetB0', 'MobileNetV2_Lite', 'MobileNetV3-Small', 'ResNet18']

# Create directories for each model
create_directories(base_dir, models)

# Save plots for each model
save_plots(history, 'EfficientNetB0', base_dir)
save_plots(history_mbv2_lite, 'MobileNetV2_Lite', base_dir)
save_plots(history_mbv3, 'MobileNetV3-Small', base_dir)
save_plots(history_resnet, 'ResNet18', base_dir)


## 7. Evaluation 

### 7.1. Metrics Calculation

In [21]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# Define the custom hard_swish activation function
def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6


In [22]:
# Load models from saved files
efficientnetb0_model = load_model('saved_models/checkpoints/efficientnetb0_retrained.h5')
mobilenetv3_model = load_model('saved_models/MobileNetV3-Small_Tuned/checkpoints/mobilenetv3_small_retrained.h5', custom_objects={'hard_swish': hard_swish})
resnet18_model = load_model('saved_models/ResNet18_Tuned/checkpoints/resnet18_retrained.h5')
mobilenetv2_lite_model = load_model('saved_models/MobileNetV2_Lite_Tuned/checkpoints/mobilenetv2_lite_retrained.h5', custom_objects={'hard_swish': hard_swish})


In [39]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, roc_curve
import numpy as np

# Define a function to calculate classification metrics
def calculate_metrics(model, generator):
    # Predict on the generator data (validation or test data)
    y_true = generator.classes
    y_pred = model.predict(generator, verbose=1)
    
    # Get the predicted class labels
    y_pred_class = np.argmax(y_pred, axis=1)

    # Calculate the metrics
    accuracy = accuracy_score(y_true, y_pred_class)
    precision = precision_score(y_true, y_pred_class, average='weighted')
    recall = recall_score(y_true, y_pred_class, average='weighted')
    f1 = f1_score(y_true, y_pred_class, average='weighted')
    try:
        auc = roc_auc_score(y_true, y_pred, multi_class='ovr')
    except:
        auc = 'N/A'  # AUC might not be available in certain cases (e.g., binary classification)

    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc
    }


In [ ]:
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define the custom hard_swish activation function (for the models using it)
def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

# Define the calculate_metrics function (assuming you have this defined)
def calculate_metrics(model, generator):
    y_true = []
    y_pred = []
    
    for i in range(len(generator)):
        X_batch, y_batch = generator[i]
        y_true.extend(np.argmax(y_batch, axis=1))  # Actual labels
        y_pred.extend(np.argmax(model.predict(X_batch), axis=1))  # Predicted labels
    
    # Calculate metrics
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, model.predict(generator, verbose=1), multi_class='ovr', average='weighted')
    
    return {
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'AUC': auc
    }

# Define the models (Make sure these models are already trained and saved as per your previous steps)
models = {
    'EfficientNetB0': efficientnetb0_model,
    'MobileNetV3-Small': mobilenetv3_model,
    'ResNet18': resnet18_model,
    'MobileNetV2-Lite': mobilenetv2_lite_model
}

# Initialize a dictionary to store results
results = {}
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    metrics = calculate_metrics(model, valid_generator)  # Replace valid_generator with your data generator
    results[model_name] = metrics

# Convert the results into a pandas DataFrame for easy comparison
results_df = pd.DataFrame(results).T  

# Display the results
print("Classification Metrics Comparison:")
print(results_df)


Evaluating EfficientNetB0...
66/66 [==============================] - 3s 49ms/step

Evaluating MobileNetV3-Small...
66/66 [==============================] - 3s 52ms/step

Evaluating ResNet18...
66/66 [==============================] - 3s 52ms/step

Evaluating MobileNetV2-Lite...
 1/66 [..............................] - ETA: 4s

c:\Users\Administrator\anaconda3\envs\py310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


66/66 [==============================] - 3s 48ms/step
Classification Metrics Comparison:
                   Precision    Recall  F1-score       AUC
EfficientNetB0      0.778904  0.650336  0.669520  0.895717
MobileNetV3-Small   0.934308  0.929395  0.930678  0.987474
ResNet18            0.854453  0.791547  0.805827  0.942046
MobileNetV2-Lite    0.670692  0.664745  0.647974  0.835977


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16888\3952211229.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2. Confusion Matrix Plotting

In [80]:
# Avoid naming conflict with keras.models
model_dict = {
    'EfficientNetB0': efficientnetb0_model,
    'MobileNetV3-Small': mobilenetv3_model,
    'ResNet18': resnet18_model,
    'MobileNetV2-Lite': mobilenetv2_lite_model
}

In [ ]:
# Print the model names in the dictionary
print("Models in model_dict:")
print(model_dict.keys())

In [61]:
def plot_confusion_matrix(model, generator, model_name, class_indices, save_dir='model_plots/confusion_matrices'):
    try:
        print(f"🔄 Predicting for {model_name}...")
        y_true = generator.classes
        y_pred_prob = model.predict(generator, verbose=1)
        y_pred = np.argmax(y_pred_prob, axis=1)

        cm = confusion_matrix(y_true, y_pred)
        class_labels = list(class_indices.keys())

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()

        os.makedirs(save_dir, exist_ok=True)
        file_path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
        plt.savefig(file_path)
        print(f"✅ Saved confusion matrix: {file_path}")
        plt.close()

    except Exception as e:
        print(f"❌ Error while plotting {model_name}: {e}")


In [65]:
for model_name, model in model_dict.items():
    print(f"🔥 Starting confusion matrix for: {model_name}")
    plot_confusion_matrix(model, valid_generator, model_name, train_generator.class_indices)


🔥 Starting confusion matrix for: EfficientNetB0
🔄 Predicting for EfficientNetB0...
61/61 [==============================] - 5s 86ms/step
✅ Saved confusion matrix: model_plots/confusion_matrices\EfficientNetB0_confusion_matrix.png
🔥 Starting confusion matrix for: MobileNetV3-Small
🔄 Predicting for MobileNetV3-Small...
61/61 [==============================] - 5s 65ms/step
✅ Saved confusion matrix: model_plots/confusion_matrices\MobileNetV3-Small_confusion_matrix.png
🔥 Starting confusion matrix for: ResNet18
🔄 Predicting for ResNet18...
61/61 [==============================] - 7s 106ms/step
✅ Saved confusion matrix: model_plots/confusion_matrices\ResNet18_confusion_matrix.png
🔥 Starting confusion matrix for: MobileNetV2-Lite
🔄 Predicting for MobileNetV2-Lite...
61/61 [==============================] - 7s 112ms/step
✅ Saved confusion matrix: model_plots/confusion_matrices\MobileNetV2-Lite_confusion_matrix.png


### 7.3. Model Comparison & Ranking

In [ ]:
# Classification results (from metrics calculation)
results = {
    'EfficientNetB0': {
        'Precision': 0.778904,
        'Recall': 0.650336,
        'F1-Score': 0.669520,
        'AUC': 0.895717
    },
    'MobileNetV3-Small': {
        'Precision': 0.934308,
        'Recall': 0.929395,
        'F1-Score': 0.930678,
        'AUC': 0.987474
    },
    'ResNet18': {
        'Precision': 0.854453,
        'Recall': 0.791547,
        'F1-Score': 0.805827,
        'AUC': 0.942046
    },
    'MobileNetV2-Lite': {
        'Precision': 0.670692,
        'Recall': 0.664745,
        'F1-Score': 0.647974,
        'AUC': 0.835977
    }
}


In [70]:
# Convert dictionary to DataFrame and transpose
results_df = pd.DataFrame(results).T

# Rank models
results_df['F1_Score_Rank'] = results_df['F1-Score'].rank(ascending=False)
results_df['AUC_Rank'] = results_df['AUC'].rank(ascending=False)

# Composite score = weighted average (70% F1, 30% AUC)
results_df['CompositeScore'] = results_df['F1_Score_Rank'] * 0.7 + results_df['AUC_Rank'] * 0.3

# Sort to find the best model
best_model_name = results_df.sort_values('CompositeScore').index[0]

# Display results
print("🏆 Ultimate Judgement:")
print(f"The best model is **{best_model_name}** based on highest F1-Score and strong AUC.\n")
print(results_df[['F1-Score', 'AUC', 'CompositeScore']].sort_values(by='CompositeScore'))


🏆 Ultimate Judgement:
The best model is **MobileNetV3-Small** based on highest F1-Score and strong AUC.

                   F1-Score       AUC  CompositeScore
MobileNetV3-Small  0.930678  0.987474             1.0
ResNet18           0.805827  0.942046             2.0
EfficientNetB0     0.669520  0.895717             3.0
MobileNetV2-Lite   0.647974  0.835977             4.0


In [72]:
results_df.to_csv("classification_model_comparison.csv")
print("✅ Results saved to classification_model_comparison.csv")

✅ Results saved to classification_model_comparison.csv


# Model Results
| Model               | Precision | Recall  | F1-score | AUC     |
|---------------------|-----------|---------|----------|---------|
| **EfficientNetB0**   | 0.778904  | 0.650336| 0.669520 | 0.895717|
| **MobileNetV3-Small**| 0.934308  | 0.929395| 0.930678 | 0.987474|
| **ResNet18**         | 0.854453  | 0.791547| 0.805827 | 0.942046|
| **MobileNetV2-Lite** | 0.670692  | 0.664745| 0.647974 | 0.835977|

### Observations:

- **MobileNetV3-Small** exhibits the best performance across all metrics, with the highest Precision (93.43%), Recall (92.94%), F1-score (93.07%), and AUC (98.75%).
- **ResNet18** provides a good balance between Precision (85.45%) and Recall (79.15%), yielding a solid F1-score (80.58%) and AUC (94.20%).
- **EfficientNetB0** has a relatively lower Recall (65.03%) and F1-score (66.95%) compared to the other models but performs decently with an AUC of 89.57%.
- **MobileNetV2-Lite** demonstrates the weakest performance in this comparison, with a Precision of 67.07%, Recall of 66.47%, F1-score of 64.80%, and AUC of 83.60%.

### Conclusion:

- **MobileNetV3-Small** is the preferred model due to its superior performance across all metrics.


## 8. Prediction 
Generate predictions on the test dataset and save results in submission CSV format.

In [32]:
# Hard Swish function for MobileNetV3-Small

def hard_swish(x):
    return x * tf.nn.relu6(x + 3) / 6

model = load_model(
    'saved_models/MobileNetV3-Small_Tuned/checkpoints/mobilenetv3_small_retrained.h5',
    custom_objects={'hard_swish': hard_swish}
)

In [33]:
# ImageDataGenerator for test data

test_dir = "test_images"
input_size = CONFIG['INPUT_SIZE']  

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    directory='.',
    classes=[test_dir],  
    target_size=input_size,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=False,
    class_mode=None
)

Found 3469 images belonging to 1 classes.


In [34]:
# Get predicted probabilities and convert to class index
pred_probs = model.predict(test_generator, verbose=1)
pred_indices = np.argmax(pred_probs, axis=1)

109/109 [==============================] - 8s 65ms/step


In [35]:
# Get mapping from integer index back to variety name
idx_to_variety = {v: k for k, v in train_generator.class_indices.items()}
pred_labels = [idx_to_variety[i] for i in pred_indices]

In [36]:
# Extract image filenames (without folders)
image_ids = [os.path.basename(path) for path in test_generator.filenames]

# Load the existing prediction_submission.csv
submission_df = pd.read_csv("prediction_submission.csv")

# Overwrite only the 'variety' column with current model predictions
submission_df['variety'] = pred_labels

# Optional: sanity check
assert len(submission_df) == len(pred_labels), "Row count mismatch!"

# Save it back
submission_df.to_csv('prediction_submission.csv', index=False)
print("✅ Updated 'variety' column in: prediction_submission.csv")


✅ Updated 'variety' column in: prediction_submission.csv
